### PakMart Traders – Incremental Data Load

Uses Delta Lake MERGE to upsert the 2023 incremental sales data into `fact_sale`.

### Read incremental CSV and create temporary view

In [ ]:
from pyspark.sql.functions import col, year, month, quarter

df = spark.read.format('csv').option('header','true').load('Files/pakmart/incremental/fact_sale_1y_incremental')
df = df.withColumn('Year', year(col('InvoiceDateKey')))
df = df.withColumn('Quarter', quarter(col('InvoiceDateKey')))
df = df.withColumn('Month', month(col('InvoiceDateKey')))
df.createOrReplaceTempView('fact_sale_incremental')

In [ ]:
# Verify incremental source — row count by period
from pyspark.sql.functions import count as _count
spark.table('fact_sale_incremental') \
     .groupBy('Year', 'Quarter', 'Month') \
     .agg(_count('*').alias('Rows')) \
     .orderBy('Year', 'Quarter', 'Month') \
     .show(20)

### Delta Lake MERGE (upsert) into silver fact_sale

The ON clause matches on `SaleKey + InvoiceDateKey` and uses partition pruning on `Year IN (2023)` to minimise the scan.

In [ ]:
%%sql
MERGE INTO fact_sale target
USING fact_sale_incremental source
ON
  source.SaleKey = target.SaleKey AND source.InvoiceDateKey = target.InvoiceDateKey
  AND target.Year IN (2023) AND target.Quarter IN (1,2,3,4)  -- Partition pruning
WHEN MATCHED THEN
  UPDATE SET
    target.CityKey            = source.CityKey,
    target.CustomerKey        = source.CustomerKey,
    target.BillToCustomerKey  = source.BillToCustomerKey,
    target.StockItemKey       = source.StockItemKey,
    target.DeliveryDateKey    = source.DeliveryDateKey,
    target.SalespersonKey     = source.SalespersonKey,
    target.WWIInvoiceID       = source.WWIInvoiceID,
    target.Description        = source.Description,
    target.Package            = source.Package,
    target.Quantity           = source.Quantity,
    target.UnitPrice          = source.UnitPrice,
    target.TaxRate            = source.TaxRate,
    target.TotalExcludingTax  = source.TotalExcludingTax,
    target.TaxAmount          = source.TaxAmount,
    target.Profit             = source.Profit,
    target.TotalIncludingTax  = source.TotalIncludingTax,
    target.TotalDryItems      = source.TotalDryItems,
    target.TotalChillerItems  = source.TotalChillerItems,
    target.LineageKey         = source.LineageKey
WHEN NOT MATCHED THEN
  INSERT (
    target.SaleKey, target.CityKey, target.CustomerKey, target.BillToCustomerKey,
    target.StockItemKey, target.InvoiceDateKey, target.DeliveryDateKey,
    target.SalespersonKey, target.WWIInvoiceID, target.Description, target.Package,
    target.Quantity, target.UnitPrice, target.TaxRate, target.TotalExcludingTax,
    target.TaxAmount, target.Profit, target.TotalIncludingTax,
    target.TotalDryItems, target.TotalChillerItems, target.LineageKey,
    target.Year, target.Quarter, target.Month)
  VALUES (
    source.SaleKey, source.CityKey, source.CustomerKey, source.BillToCustomerKey,
    source.StockItemKey, source.InvoiceDateKey, source.DeliveryDateKey,
    source.SalespersonKey, source.WWIInvoiceID, source.Description, source.Package,
    source.Quantity, source.UnitPrice, source.TaxRate, source.TotalExcludingTax,
    source.TaxAmount, source.Profit, source.TotalIncludingTax,
    source.TotalDryItems, source.TotalChillerItems, source.LineageKey,
    source.Year, source.Quarter, source.Month)

In [ ]:
# Verify fact_sale after merge
from pyspark.sql.functions import count as _count
spark.table('fact_sale') \
     .groupBy('Year', 'Quarter', 'Month') \
     .agg(_count('*').alias('Rows')) \
     .orderBy('Year', 'Quarter', 'Month') \
     .show(60)

In [ ]:
# Show Delta history for fact_sale
from delta.tables import DeltaTable
DeltaTable.forName(spark, 'fact_sale').history().select(
    'version', 'timestamp', 'operation', 'operationParameters'
).show(10, truncate=60)